### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

#### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

groq_llm = ChatGroq(model="qwen/qwen3.8-27b")

groq_llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1'}}, client=<groq.resources.chat.completions.Completions object at 0x000002290BDC7770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002290BF70590>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [38]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="Title of the Movie")
    year:int=Field(description="Year in which movie released")
    director:str=Field(description="Director of the movie")
    rating:float=Field(description="Rating of the movie out of 10")

In [39]:
structured_output = groq_llm.with_structured_output(Movie)
structured_output


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1'}}, client=<groq.resources.chat.completions.Completions object at 0x000002290BDC7770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002290BF70590>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the Movie', 'type': 'string'}, 'year': {'description': 'Year in which movie released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': 'Rating of the movie out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'desc

Without Output Structure

In [37]:
groq_llm.invoke("Bahubali").text

'**Bahubali** (also known as *Baahubali*) is a highly acclaimed Indian epic fantasy film series directed by **S. S. Rajamouli** and produced by **D. Ramana**. The films are celebrated for their groundbreaking visual effects, grand scale, powerful storytelling, and iconic music.\n\n### Key Details:\n\n#### **1. The Films**\n- **Baahubali: The Beginning** (2015)\n- **Baahubali: The Conclusion** (2017)\n\nBoth films were released in Telugu and Tamil, with Hindi versions also achieving massive box office success. Together, they became one of the highest-grossing Indian film franchises of all time.\n\n#### **2. Plot Summary**\nThe story is an epic retelling of the legendary tale of **Amarendra Baahubali**, a warrior-prince of the kingdom of Mahishmati.\n\n- **The Beginning**: Introduces Shivudu (played by Prabhas), a humble servant in the village of Kotha, who learns he is the son of the late King Baahubali. His journey begins to uncover the truth about his father’s legacy and his own royal

With Output Structure

In [40]:
response=structured_output.invoke("Bahubali")
print(response)
print(response.title)
print(response.director)
print(response.year)
print(response.rating)

title='Bahubali: The Beginning' year=2015 director='S. S. Rajamouli' rating=8.1
Bahubali: The Beginning
S. S. Rajamouli
2015
8.1


Message output alongside parsed structure

In [41]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = groq_llm.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Baahubali: The Conclusion")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rgpsspfag', 'function': {'arguments': '{"director":"S. S. Rajamouli","rating":8.4,"title":"Baahubali: The Conclusion","year":2017}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 360, 'total_tokens': 439, 'completion_time': 0.238215533, 'completion_tokens_details': None, 'prompt_time': 0.030894485, 'prompt_tokens_details': None, 'queue_time': 0.064693604, 'total_time': 0.269110018}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b4a9-8fe2-7d90-bedc-70b7e306e038-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'S. S. Rajamouli', 'rating': 8.4, 'title': 'Baahubali: The Conclusion', 'year': 2017}, 'id': 'rgpsspfag', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 360,

Nested Structure

In [76]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: str = Field(description="give month in text,year of movie release")
    cast: list[Actor] = Field(description="Top 4 Actors only") # Above class is used as Each Actor will have name and role here
    genres: list[str]
    budget: float | None = Field(None, description="Budget in IND Rupees in Crores")

model_with_structure = groq_llm.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Baahubali: The beginning")

print(response.title)
print(response.year)
print(response.cast)
print(response.genres)
print(response.budget)

Baahubali: The Beginning
July 2015
[Actor(name='Prabhas', role='Amarendra Baahubali'), Actor(name='Rana Daggubati', role='Bhallaladeva'), Actor(name='Anushka Shetty', role='Devasena'), Actor(name='Tamannaah Bhatia', role='Avanthika')]
['Action', 'Drama', 'Adventure']
180.0


#### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [79]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=groq_llm.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie RRR")
response

{'director': 'S. S. Rajamouli', 'rating': 8.4, 'title': 'RRR', 'year': 2022}

It will not follow the runtime validation

In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor] = Field(description="Top 4 Actors only")  # but still it is giving more than 4 actors
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Crore Rupees") 

model_with_structure = groq_llm.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie RRR")
response

{'budget': 130000000,
 'cast': [{'name': 'N.T. Rama Rao Jr.', 'role': 'Amarendra Baahu'},
  {'name': 'Ram Charan', 'role': 'Commaran Bhanu'},
  {'name': 'Alia Bhatt', 'role': 'Elangu'},
  {'name': 'Ajay Devgn', 'role': 'Sirocco'},
  {'name': 'Rana Daggubati', 'role': 'Bhairava Rao'},
  {'name': 'Samantha Ruth Prabhu', 'role': 'Jhansi Lakshmi'},
  {'name': 'Oscar Leight', 'role': 'Robert'}],
 'genres': ['Action', 'Drama', 'Historical'],
 'title': 'RRR',
 'year': 2022}

#### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [94]:
#Pydantic

from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=ChatGroq(model="qwen/qwen3.8-27b"),
    response_format= ContactInfo
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Abhijit Deshmukh, abhijit@example.com, (+91) 8479808281, Kundal"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: Abhijit Deshmukh, abhijit@example.com, (+91) 8479808281, Kundal', additional_kwargs={}, response_metadata={}, id='b93cd5fc-7330-4f3d-94b6-9310d21afaa9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ysz4ar1gn', 'function': {'arguments': '{"email":"abhijit@example.com","name":"Abhijit Deshmukh","phone":"(+91) 8479808281"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 364, 'total_tokens': 438, 'completion_time': 0.266353641, 'completion_tokens_details': None, 'prompt_time': 0.025915847, 'prompt_tokens_details': None, 'queue_time': 0.060202512, 'total_time': 0.292269488}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b4eb-1d47-7ad3-8e5e-6f489d30a776-0', tool_calls=[{'name': 'Contact

In [95]:
result["structured_response"]

ContactInfo(name='Abhijit Deshmukh', email='abhijit@example.com', phone='(+91) 8479808281')

In [98]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=ChatGroq(model="qwen/qwen3.8-27b"),
    response_format= ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Abhijit Deshmukh, abhijit@example.com, (+91) 8479808281, Kundal"}]
})

result["structured_response"]

ContactInfo(name='Abhijit Deshmukh', email='abhijit@example.com', phone='(+91) 8479808281')